In [1]:
import nilearn as nl
import numpy as np
import matplotlib
import pandas as pd
import os
import nibabel as nib
import matplotlib.pyplot as plt
import openpyxl as xl

from nilearn.glm.first_level import make_first_level_design_matrix
from nilearn.plotting import plot_design_matrix, plot_contrast_matrix
from nilearn.glm.first_level import FirstLevelModel
from nilearn.masking import compute_epi_mask, apply_mask, unmask
from nilearn.image import index_img, resample_to_img, math_img, mean_img, get_data
from nilearn.plotting import plot_stat_map, show
from nilearn import image, masking
from nilearn import plotting
from nilearn.datasets import load_mni152_brain_mask
from pandas import read_excel

from nilearn.glm.second_level import SecondLevelModel
from scipy.stats import norm

from nilearn.glm import threshold_stats_img
from pathlib import Path

# fcontrast - test moral vs non-moral for each story separately

In [2]:
virtueORvice = 'virtues'

stories = ["tunnel","21styear"]
# stories = ["tunnel","21styear","prettymouth","shapessocial","shapesphysical"]


# output directory
base_dir1 = os.path.expanduser(
    "~/Library/CloudStorage/GoogleDrive-s.h.p.collin@tilburguniversity.edu/My Drive/research/ongoing/2024_shapesNarrativeMRI/__narratives_and_values_fMRI/wholeBrain_results/result_interpretation/"
)

output_dir = Path(f"{base_dir1}{virtueORvice}/fcontrast/")


In [3]:

for story in stories:
    print(f"\nProcessing story: {story}")
    
    if virtueORvice == 'virtues':
        virtueORviceDataFolder = '7_MAC'
    
    if virtueORvice == 'vices':
        virtueORviceDataFolder = '7_MAC_V'
    
    base_dir2 = os.path.expanduser(
        "~/Library/CloudStorage/GoogleDrive-s.h.p.collin@tilburguniversity.edu/My Drive/research/ongoing/2024_shapesNarrativeMRI/__narratives_and_values_fMRI/wholeBrain_results/processed_first_level_per_sentence/"
    )
    
    if story == "milkyway":
        print("Milky Way combining...")
        subfolders = [
            Path(base_dir2) / "milkywaysynonyms" / virtueORviceDataFolder / "F_contrast",
            Path(base_dir2) / "milkywayoriginal" / virtueORviceDataFolder / "F_contrast",
            Path(base_dir2) / "milkywayvodka" / virtueORviceDataFolder / "F_contrast",
        ]

        full_paths = []
        for subfolder in subfolders:
            if subfolder.exists():
                files = [f for f in os.listdir(subfolder) if f.endswith((".nii", ".nii.gz"))]
                full_paths.extend([subfolder / f for f in files])
            else:
                print(f"Warning: subfolder not found — {subfolder}")
    else:
        subfolder = Path(base_dir2) / story / virtueORviceDataFolder / "F_contrast"
        if subfolder.exists():
            files = [f for f in os.listdir(subfolder) if f.endswith((".nii", ".nii.gz"))]
            full_paths = [subfolder / f for f in files]
        else:
            print(f"Warning: subfolder not found — {subfolder}")
            full_paths = []
    
    # Sanity check
    if not full_paths:
        print(f"No .nii or .nii.gz files found in {subfolder}. Skipping...")
        continue

    full_paths = [str(p) for p in full_paths]
    
    # Create design matrix
    design_matrix = pd.DataFrame(
        [1] * len(full_paths),
        columns=["intercept"],
    )

    # Fit the second-level model
    second_level_model = SecondLevelModel()
    second_level_model = second_level_model.fit(
        full_paths,
        design_matrix=design_matrix,
    )

    # Compute z-map
    z_map = second_level_model.compute_contrast(
        second_level_contrast="intercept",
        output_type="z_score",
    )

    # Save z-map in subfolder
    z_map_filename = os.path.join(output_dir, f"{story}/zscores_fcontrast_secondlevel.nii.gz")
    z_map.to_filename(z_map_filename)
    print(f"✅ Saved z-map to: {z_map_filename}")

    # Threshold map
    thresholded_map, threshold = threshold_stats_img(
        stat_img=z_map,
        alpha=0.05,
        height_control='bonferroni', #bonferroni,fdr
        cluster_threshold=0, #2,5,10,0
        two_sided=True,
    )

    # Save thresholded map in subfolder
    thresh_filename = os.path.join(
        output_dir,
        f"{story}/threshold_{threshold:.3g}_bonfcorrect_zscores_fcontrast_secondlevel.nii.gz",
    )
    thresholded_map.to_filename(thresh_filename)
    print(f"✅ Saved thresholded map to: {thresh_filename}")


    # Threshold map one-sided
    thresholded_map, threshold = threshold_stats_img(
        stat_img=z_map,
        alpha=0.05,
        height_control='bonferroni', #bonferroni,fdr
        cluster_threshold=0, #2,5,10,0
        two_sided=False,
    )

    # Save thresholded map in subfolder
    thresh_filename = os.path.join(
        output_dir,
        f"{story}/threshold_{threshold:.3g}_bonfcorrectOneSided_zscores_fcontrast_secondlevel.nii.gz",
    )
    thresholded_map.to_filename(thresh_filename)
    print(f"✅ Saved thresholded map to: {thresh_filename}")



Processing story: tunnel
✅ Saved z-map to: /Users/silvycollin/Library/CloudStorage/GoogleDrive-s.h.p.collin@tilburguniversity.edu/My Drive/research/ongoing/2024_shapesNarrativeMRI/__narratives_and_values_fMRI/wholeBrain_results/result_interpretation/virtues/fcontrast/tunnel/zscores_fcontrast_secondlevel.nii.gz
✅ Saved thresholded map to: /Users/silvycollin/Library/CloudStorage/GoogleDrive-s.h.p.collin@tilburguniversity.edu/My Drive/research/ongoing/2024_shapesNarrativeMRI/__narratives_and_values_fMRI/wholeBrain_results/result_interpretation/virtues/fcontrast/tunnel/threshold_4.91_bonfcorrect_zscores_fcontrast_secondlevel.nii.gz
✅ Saved thresholded map to: /Users/silvycollin/Library/CloudStorage/GoogleDrive-s.h.p.collin@tilburguniversity.edu/My Drive/research/ongoing/2024_shapesNarrativeMRI/__narratives_and_values_fMRI/wholeBrain_results/result_interpretation/virtues/fcontrast/tunnel/threshold_4.77_bonfcorrectOneSided_zscores_fcontrast_secondlevel.nii.gz

Processing story: 21styear
✅ S